# 04 — Statistical Process Control on Predictions and Sensors

Apply three classic SPC methods to the flotation plant streams:

1. **Shewhart** with Western Electric rules (rules 1–4) — sensitive to single-point excursions.
2. **CUSUM** — accumulates deviations from a target; detects small persistent shifts.
3. **EWMA** — exponentially weighted moving average; detects medium-sized shifts.

Why all three: they have different sensitivities. Shewhart catches sudden spikes; CUSUM catches a 0.5σ shift that lasts hours; EWMA is the middle ground. **Production SPC dashboards use all three together.**

**This notebook applies SPC to the residuals of the LightGBM model.** When the model starts being **wrong systematically** in test, that is the early signal of distribution shift — exactly what we want to catch. The notebook uses the fresh-only model from notebook 02b if available; otherwise falls back to the all-rows model from notebook 02.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import TARGET_COLS, temporal_split, detect_constant_lab_measurements
from frothiq.features.pipeline import load_gold, list_feature_cols
from frothiq.models.spc import (
    CusumParams, EwmaParams,
    fit_control_limits, western_electric_violations,
    cusum_chart, ewma_chart,
)

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')

## 1. Load gold features + the LightGBM model

Prefer the fresh-only model from notebook 02b. Fall back to the all-rows model from notebook 02 if the fresh one isn't registered yet.

In [ ]:
gold = load_gold(ROOT / 'data' / 'processed' / 'gold.parquet')
feature_cols = list_feature_cols(gold, target_cols=TARGET_COLS)

# Use the same fresh subset for SPC analysis if the fresh-only model is being used.
# This avoids comparing apples and oranges (model trained on fresh, evaluated on stale).
fresh_mask = detect_constant_lab_measurements(gold)
gold_fresh = gold[fresh_mask].reset_index(drop=True)
_, _, test = temporal_split(gold_fresh, train_frac=0.7, val_frac=0.15)
test = test.dropna(subset=feature_cols + TARGET_COLS).reset_index(drop=True)
print(f'Test rows (fresh subset): {len(test):,}')

In [ ]:
import mlflow.lightgbm

models = {}
model_source = None
for experiment_name in ['frothiq-baseline-fresh', 'frothiq-baseline']:
    if all(t in models for t in TARGET_COLS):
        break
    for target in TARGET_COLS:
        if target in models:
            continue
        runs = mlflow.search_runs(
            experiment_names=[experiment_name],
            filter_string=f"params.target = '{target}'",
            order_by=['start_time DESC'],
            max_results=1,
        )
        if not len(runs):
            continue
        run_id = runs.iloc[0]['run_id']
        model_uri = f'runs:/{run_id}/model'
        models[target] = mlflow.lightgbm.load_model(model_uri)
        if model_source is None:
            model_source = experiment_name
        print(f'{target}: loaded from {experiment_name} ({model_uri})')

if not models:
    raise RuntimeError('No LightGBM run found. Run notebook 02 (or 02b) first.')
print(f'\nModel source: {model_source}')

In [ ]:
# Predict on the test set + compute residuals.
for target, model in models.items():
    test[f'pred_{target}'] = model.predict(test[feature_cols])
    test[f'resid_{target}'] = test[target] - test[f'pred_{target}']
test[['timestamp'] + [c for c in test.columns if c.startswith(('pred_', 'resid_'))]].head()

## 2. Shewhart with Western Electric rules

Fit control limits on the **first 20% of test residuals** (assumed in-control), apply rules over the remaining 80%.

In [ ]:
target = 'pct_iron_concentrate'
resid = test[f'resid_{target}'].dropna().values
n_baseline = max(100, int(len(resid) * 0.2))
limits = fit_control_limits(resid[:n_baseline])
print(f'Baseline center: {limits.center:.4f}, σ: {limits.sigma:.4f}')
print(f'Total test rows: {len(resid):,}, baseline rows: {n_baseline:,}')

violations = western_electric_violations(resid, limits)
for rule, flags in violations.items():
    print(f'{rule}: {flags.sum()} violations ({100 * flags.mean():.2f}%)')

In [ ]:
# Plot Shewhart chart with all 4 WE rules highlighted.
any_violation = (
    violations['rule_1'] | violations['rule_2']
    | violations['rule_3'] | violations['rule_4']
)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(resid, color='#1f77b4', alpha=0.7, lw=0.8, label='residual')
ax.axhline(limits.center, color='green', ls=':', label='μ (baseline)')
ax.axhline(limits.ucl_3, color='red', ls='--', alpha=0.7, label='±3σ (baseline)')
ax.axhline(limits.lcl_3, color='red', ls='--', alpha=0.7)
ax.axvline(n_baseline, color='gray', ls=':', alpha=0.5, label='end of baseline')
ax.scatter(
    np.where(any_violation)[0], resid[any_violation],
    color='red', marker='x', s=30, label=f'WE violations ({any_violation.sum()})',
)
ax.set_title(f'Shewhart chart with Western Electric rules — residuals of {target}')
ax.set_xlabel('test row index (chronological)'); ax.set_ylabel('residual'); ax.legend(loc='best')
fig.tight_layout()

## 3. CUSUM

Detect a 1σ shift with decision interval h = 4σ. Standard parameters from Page 1954.

In [ ]:
params_cusum = CusumParams(
    target=limits.center, sigma=limits.sigma,
    delta_sigma=1.0, h_sigma=4.0,
)
chart_c = cusum_chart(resid, params_cusum)
print(f"CUSUM signals: {chart_c['signal'].sum()} ({100 * chart_c['signal'].mean():.2f}%)")
print(f"  Upward signals (Cu > h): {chart_c['signal_up'].sum()}")
print(f"  Downward signals (Cl > h): {chart_c['signal_down'].sum()}")

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(chart_c['cu'], label='Cu (upward)', color='#1f77b4')
ax.plot(chart_c['cl'], label='Cl (downward)', color='#ff7f0e')
ax.axhline(params_cusum.h, color='red', ls='--', alpha=0.7, label=f'h = {params_cusum.h:.3f}')
ax.axvline(n_baseline, color='gray', ls=':', alpha=0.5, label='end of baseline')
ax.set_title(f'CUSUM chart — residuals of {target}')
ax.set_xlabel('test row index (chronological)'); ax.set_ylabel('cumulative sum'); ax.legend(loc='best')
fig.tight_layout()

## 4. EWMA

λ = 0.2 (smoothing) and L = 3 (3σ-equivalent control limits). Detects medium-sized shifts.

In [ ]:
params_ewma = EwmaParams(
    target=limits.center, sigma=limits.sigma,
    lambda_=0.2, L=3.0,
)
chart_e = ewma_chart(resid, params_ewma)
print(f"EWMA signals: {chart_e['signal'].sum()} ({100 * chart_e['signal'].mean():.2f}%)")

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(resid, color='lightgray', lw=0.6, alpha=0.7, label='raw residual')
ax.plot(chart_e['z'], color='#1f77b4', label='EWMA')
ax.plot(chart_e['ucl'], color='red', ls='--', alpha=0.7, label='UCL')
ax.plot(chart_e['lcl'], color='red', ls='--', alpha=0.7, label='LCL')
ax.axvline(n_baseline, color='gray', ls=':', alpha=0.5, label='end of baseline')
ax.set_title(f'EWMA chart — residuals of {target}')
ax.set_xlabel('test row index (chronological)'); ax.set_ylabel('residual / EWMA'); ax.legend(loc='best')
fig.tight_layout()

## 5. Method comparison

Summary of which method detected what.

In [ ]:
summary = pd.DataFrame([
    {'method': 'Shewhart (rule 1: ±3σ)', 'signals': int(violations['rule_1'].sum()), 'pct': 100 * violations['rule_1'].mean()},
    {'method': 'Shewhart (rule 2: 2 of 3 ±2σ)', 'signals': int(violations['rule_2'].sum()), 'pct': 100 * violations['rule_2'].mean()},
    {'method': 'Shewhart (rule 3: 4 of 5 ±1σ)', 'signals': int(violations['rule_3'].sum()), 'pct': 100 * violations['rule_3'].mean()},
    {'method': 'Shewhart (rule 4: 8 same side)', 'signals': int(violations['rule_4'].sum()), 'pct': 100 * violations['rule_4'].mean()},
    {'method': 'CUSUM (δ=1σ, h=4σ)', 'signals': int(chart_c['signal'].sum()), 'pct': 100 * chart_c['signal'].mean()},
    {'method': 'EWMA (λ=0.2, L=3)', 'signals': int(chart_e['signal'].sum()), 'pct': 100 * chart_e['signal'].mean()},
])
summary.round(2)

## 6. What this is showing about the model

If the SPC charts light up heavily after the **end of baseline** marker, that is **not noise** — it is the model degrading systematically as the test data drifts away from the train distribution.

**Three concrete actions this enables in production:**

1. **Auto-trigger retraining** when CUSUM signals persist for N hours.
2. **Switch to a fall-back model** (or last-known-good) until retraining completes.
3. **Alert the operations team** with the type of warning (sudden Shewhart vs slow CUSUM) so they can correlate with operational changes (feed source switch, recipe adjustment, instrument recalibration).

## What's next: `05_whatif.ipynb`

Take the operator's current sensor state and the trained model. Ask: *"what if I change pH from 9.4 to 9.8?"* — show the predicted Δ on % Iron Concentrate.